# Example ETC Calculation

A cubesim exposure-time calculation combines an IFU configuration, a source model, a PSF, and an exposure sequence. In this example, the source is an inclined emission-line galaxy with an exponential Sersic profile and a rotating-disk velocity field that share the same projected geometry.

In [ ]:
import astropy.units as u

import cubesim
import cubesim.diagnostics as diagnostics
import cubesim.plotting as plotting
import inclined_galaxy

## Instrument Configuration

`cubesim.Etc` loads the instrument definition from its data directory. `inclined_galaxy.configure()` selects the 50 mas scale, the R = 3000 H+K disperser, the airmass 1.0 atmosphere, and the synthetic PSF before adding the target, exposure, and apertures. Four 600 s target exposures use the default AB nodding sequence. `include_models=True` retains the intermediate source and background models for inspection.

In [ ]:
etc = cubesim.Etc("instrument_data")
inclined_galaxy.configure(etc)
result = etc.run(include_models=True)

print(f"S/N cube shape: {result.snr.shape}")

## PSF

The normalized image in `result.psf.data` is the spatial kernel applied to the galaxy model. PSF measurements can be included in the plot title; the synthetic direct PSF has no wavelength or pupil metadata, so no Strehl ratio is shown. `radius` limits the displayed field without changing the calculation.

In [ ]:
psf_stats = diagnostics.psf_stats(result.psf, ee_apertures=[100] * u.mas)
psf_figure = plotting.plot_psf(result, stats=psf_stats, radius=0.3 * u.arcsec)

## Target Models

`result.models.targets` contains the retained source-model stages for each target. `plot_target_models()` shows the detector-resolution spatial profile, spectrum, and velocity field used to construct the combined target cube.

In [ ]:
target_models_figure = plotting.plot_target_models(result)

## Background Models

Atmospheric transmission, atmospheric sky radiance, and instrument thermal radiance are evaluated on the detector wavelength grid. `plot_background_models()` places these inputs on aligned wavelength axes for direct inspection.

In [ ]:
background_models_figure = plotting.plot_background_models(result)

## Signal Components

`result.signals` separates the detected target, sky, thermal, dark, and total electrons. A wavelength range passed to `plot_signal_components()` produces maps summed over that interval; a detector-position range produces spectra summed over those spaxels. The map isolates the emission line, while the spectrum uses the central 2 x 2 spaxels across the H+K setting.

In [ ]:
signal_maps_figure = plotting.plot_signal_components(
    result, wavelength=(2.1995, 2.2005) * u.micron
)

In [ ]:
signal_spectra_figure = plotting.plot_signal_components(
    result,
    position=((19, 20), (19, 20)),
    wavelength=(2.1, 2.3) * u.micron,
)

## S/N

`plot_snr()` accepts the same spatial and wavelength selectors as the signal plots. Range selections sum the target signal and propagate the corresponding variance before calculating S/N. The spectrum uses the central 2 x 2 spaxels, and the map integrates the narrow interval around the emission line.

In [ ]:
snr_spectrum_figure = plotting.plot_snr(
    result,
    position=((19, 20), (19, 20)),
    wavelength=(2.1, 2.3) * u.micron,
)

In [ ]:
snr_map_figure = plotting.plot_snr(
    result,
    wavelength=(2.1995, 2.2005) * u.micron,
)

## Aperture Diagnostics

`Etc.add_aperture()` registers reusable three-dimensional masks before the calculation. Each entry in `result.apertures` retains its integrated S/N together with spectral and spatial S/N projections. Passing those apertures to `plot_snr()` outlines their spatial footprints, while `plot_aperture_snr()` compares their retained projections using common display ranges.

In [ ]:
for aperture in result.apertures:
    print(f"{aperture.name}: S/N = {aperture.snr:.2f}")

In [ ]:
aperture_overview_figure = plotting.plot_snr(
    result,
    apertures=result.apertures,
    title="Aperture locations",
    cbar_range=(0, 50),
)

In [ ]:
aperture_snr_figure = plotting.plot_aperture_snr(
    result,
    aperture=result.apertures,
    cbar_range=(0, 50),
    y_range=(0, 100),
)

## Save Results

`EtcResult.save()` writes the wavelength, S/N, target-signal, background, and total-variance products to FITS extensions with their units and coordinate metadata. `overwrite=True` allows the calculation to be rerun using the same output path.

In [ ]:
result.save("etc_result.fits", overwrite=True)